## 03b: Wearable Feature Extraction

Computes the four wearable phenotypes for each participant and task, over the 90 days
preceding the test (`etm_date − 90 d` to `etm_date − 1 d`; the test day is excluded):

| Phenotype | Column | Definition |
|---|---|---|
| Steps | `mean_steps` | mean daily steps over valid days |
| WASO | `mean_waso` | mean wake after sleep onset (min) over valid nights |
| TST | `mean_sleep_duration` | mean total sleep time (min) over valid nights |
| STV | `STV` | sleep timing variability (Task 1, Task 3); requires ≥15 consecutive night pairs |

A participant is wearable-eligible for a task with ≥30 valid days and ≥30 valid nights in
that window. Photoperiod (day length at the window midpoint) is computed from the latitude
derived in `02_covariates`.

Output: `features/wearable_features.parquet`, one row per participant and task (MDD, BD and HC).

## Task 0: Setup

In [ ]:
import sys
import os
import gc
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils import *

OVERWRITE = False

# -- Constants -----------------------------------------------------------------
N_DAYS_WEARABLE  = 90   # calendar length of per-task feature window
N_REC_DAYS_MIN   = 30   # min valid days  for step features
N_REC_NIGHTS_MIN = 30   # min valid nights for sleep/WASO features
STV_PAIRS_MIN    = 15   # min consecutive valid-night pairs for STV
ETM_TASKS = ['gradcpt', 'delaydiscounting', 'flanker', 'emorecog']

# -- Input / output paths -----------------------------------------------------
VDAYS_IN  = 'raw_pulls/valid_days.parquet'
VN_IN     = 'raw_pulls/valid_nights.parquet'
FEATS_OUT = 'features/wearable_features.parquet'

# -- Cohort spine: MDD, BD, and HC — with etm_date_{task} columns and latitude -
cohort          = load_parquet('cohort/cohort_master.parquet')
cohort_wearable = cohort[cohort['group'].isin(['MDD', 'BD', 'HC'])].copy()

date_cols = [f'etm_date_{t}' for t in ETM_TASKS] + ['etm_date_first']
for col in date_cols:
    cohort_wearable[col] = pd.to_datetime(cohort_wearable[col])

if 'latitude' not in cohort_wearable.columns:
    raise RuntimeError(
        "'latitude' column missing from cohort_master — run 02_covariates first."
    )

# -- Seed incremental attrition log -------------------------------------------
# This notebook's rows are dropped and regenerated below; other notebooks' rows are kept.
NB_ATTRITION_PREFIXES = ('NB03b informational: ',)

attrition_rows.clear()
if gcs_exists('cohort/attrition.parquet'):
    _atr = load_parquet('cohort/attrition.parquet')
    _own = (_atr['step'].astype(str)
                        .map(lambda s: s.startswith(NB_ATTRITION_PREFIXES))
                        .astype(bool))
    attrition_rows.extend(_atr[~_own].to_dict('records'))
    print(f'Attrition log: kept {len(attrition_rows)} step(s) from other notebooks; '
          f'dropped {int(_own.sum())} own row(s) for regeneration.')

n_mdd = int((cohort_wearable['group'] == 'MDD').sum())
n_bd  = int((cohort_wearable['group'] == 'BD').sum())
n_hc  = int((cohort_wearable['group'] == 'HC').sum())
print(f'MDD + BD + HC participants in spine : {len(cohort_wearable):,}  '
      f'(MDD: {n_mdd:,}, BD: {n_bd:,}, HC: {n_hc:,})')
print(f'OVERWRITE                           : {OVERWRITE}')
for task in ETM_TASKS:
    n_m = int((cohort_wearable[f'mdd_eligible_{task}'].fillna(0) == 1).sum())
    n_b = int((cohort_wearable[f'bd_eligible_{task}'].fillna(0) == 1).sum())
    n_h = int(((cohort_wearable['group'] == 'HC') & cohort_wearable[f'etm_date_{task}'].notna()
               & (cohort_wearable[f'fitbit_sufficient_{task}'].fillna(0) == 1)).sum())
    print(f'  {task}: MDD {n_m:,} eligible, BD {n_b:,} eligible, HC {n_h:,} Fitbit-sufficient')


## Task 1: Noon-Anchored Sleep Timing

Onset and offset times are converted to noon-anchored decimal hours, so that nights spanning
midnight do not wrap around. Onset is encoded directly: 12:00–23:59 map to 12–24 h and
00:00–11:59 to 24–36 h (an onset at 23:00 is 23.0, one at 01:00 is 25.0). Offset (wake time) is
onset plus the elapsed time from onset to wake, so it stays on the same scale whatever the clock
time of waking (a 12:30 wake after a 23:00 onset is 36.5). Encoding the wake time on its own
would wrap late wake times by 24 h.

In [ ]:
# -- Load inputs --------------------------------------------------------------
valid_days = load_parquet(VDAYS_IN)
valid_days['date'] = pd.to_datetime(valid_days['date'])
print(f'valid_days  : {len(valid_days):,} rows, {valid_days["person_id"].nunique():,} participants')

vn = load_parquet(VN_IN)
for col in ['night_date', 'onset_dt', 'offset_dt']:
    vn[col] = pd.to_datetime(vn[col])
print(f'valid_nights: {len(vn):,} rows, {vn["person_id"].nunique():,} participants')

# -- Noon-anchor encoding (computed once over the full union window) ----------
# Onset is encoded directly (23:00 -> 23.0, 01:00 -> 25.0). Offset is onset plus the
# elapsed time from onset to wake, so it stays on the same scale (a 12:30 wake after a
# 23:00 onset -> 36.5); encoding the wake time on its own would wrap late wake times.
vn['elapsed_h']   = (vn['offset_dt'] - vn['onset_dt']).dt.total_seconds() / 3600
vn['onset_h']     = (vn['onset_dt'].dt.hour + vn['onset_dt'].dt.minute / 60
                     + vn['onset_dt'].dt.second / 3600)
vn['onset_anch']  = np.where(vn['onset_h'] >= 12, vn['onset_h'], vn['onset_h'] + 24)
vn['offset_anch'] = vn['onset_anch'] + vn['elapsed_h']

# -- Encoding QC (against the 03a night gates) ---------------------------------
assert ((vn['onset_anch'] >= 18) & (vn['onset_anch'] < 32)).all(), \
    'valid nights have onset between 18:00 and 08:00 (03a onset gate)'
assert ((vn['elapsed_h'] > 0) & (vn['elapsed_h'] <= 12)).all(), \
    'valid nights have 0 < onset to offset <= 720 min (03a episode gate)'
late_wake = vn['offset_anch'] >= 42   # wake at or after 18:00 on the day after the night
print(f'\nNights with wake at or after 18:00 (would wrap by 24 h if the wake time were '
      f'encoded on its own): {late_wake.sum():,} ({100*late_wake.mean():.3f}%), '
      f'{vn.loc[late_wake, "person_id"].nunique():,} participants')

# -- Descriptive stats on anchored onset ---------------------------------------
print(f'\nonset_anch  : min={vn["onset_anch"].min():.2f}  '
      f'median={vn["onset_anch"].median():.2f}  max={vn["onset_anch"].max():.2f}')
print(f'offset_anch : min={vn["offset_anch"].min():.2f}  '
      f'median={vn["offset_anch"].median():.2f}  max={vn["offset_anch"].max():.2f}')


## Task 2: Photoperiod

Astronomical day length in hours (Spencer, 1971), at the midpoint of each task's window.
Missing latitude gives a missing photoperiod.

In [ ]:
def daylength_vec(latitude, jday):
    """Astronomical day length (hours). Spencer (1971) / Almorox (2005). Vectorized."""
    lat   = np.asarray(latitude, dtype=float)
    jd    = np.asarray(jday,     dtype=float)
    gamma = 2 * np.pi / 365 * (jd - 1)
    delta = (180 / np.pi) * (
        0.006918
        - 0.399912 * np.cos(gamma)   + 0.070257 * np.sin(gamma)
        - 0.006758 * np.cos(2*gamma) + 0.000907 * np.sin(2*gamma)
        - 0.002697 * np.cos(3*gamma) + 0.001480 * np.sin(3*gamma)
    )
    cos_wo = (
        np.sin(-0.8333 / 360 * 2*np.pi)
        - np.sin(lat/360*2*np.pi) * np.sin(delta/360*2*np.pi)
    ) / (np.cos(lat/360*2*np.pi) * np.cos(delta/360*2*np.pi))
    return np.where(
        cos_wo > 1,  0.0,
        np.where(
            cos_wo < -1, 24.0,
            2 * np.arccos(np.clip(cos_wo, -1.0, 1.0)) / (15/360*2*np.pi)
        )
    )

## Task 3: Per-Task Features

For each task, valid days and nights are restricted to the 90-day window and the phenotypes
computed. STV is the mean of the mean absolute successive differences of onset and offset
times, over strictly consecutive valid nights:
`STV = (MASD_onset + MASD_offset) / 2`.

In [ ]:
results = []

for task in ETM_TASKS:
    print(f'\n=== {task} ===')

    # -- Per-participant task window ------------------------------------------
    # Include MDD-eligible, BD-eligible, and HC with Fitbit-sufficient coverage
    eligible_mask = (
        ((cohort_wearable['group'] == 'MDD') & (cohort_wearable[f'mdd_eligible_{task}'].fillna(0) == 1)) |
        ((cohort_wearable['group'] == 'BD')  & (cohort_wearable[f'bd_eligible_{task}'].fillna(0) == 1)) |
        ((cohort_wearable['group'] == 'HC')  & cohort_wearable[f'etm_date_{task}'].notna()
                                             & (cohort_wearable[f'fitbit_sufficient_{task}'].fillna(0) == 1))
    )
    tw = (
        cohort_wearable[eligible_mask]
        [['person_id', 'group', f'etm_date_{task}', 'latitude']]
        .dropna(subset=[f'etm_date_{task}'])
        .copy()
        .rename(columns={f'etm_date_{task}': 'etm_date'})
    )
    # 90 days before the test, test day excluded
    tw['t0']    = tw['etm_date'] - pd.Timedelta(days=N_DAYS_WEARABLE)
    tw['t_end'] = tw['etm_date'] - pd.Timedelta(days=1)
    pid_set     = set(tw['person_id'])
    t0_map_k    = tw.set_index('person_id')['t0'].to_dict()
    t_end_map_k = tw.set_index('person_id')['t_end'].to_dict()
    lat_map_k   = tw.set_index('person_id')['latitude'].to_dict()
    n_mdd_task = int((tw['group'] == 'MDD').sum())
    n_bd_task  = int((tw['group'] == 'BD').sum())
    n_hc_task  = int((tw['group'] == 'HC').sum())
    print(f'  {len(tw):,} eligible completers (MDD: {n_mdd_task:,}, BD: {n_bd_task:,}, HC: {n_hc_task:,})')

    # -- Filter valid_days to [t0_k, t_end_k] --------------------------------
    vd_t = valid_days[valid_days['person_id'].isin(pid_set)].copy()
    vd_t['_t0']    = vd_t['person_id'].map(t0_map_k)
    vd_t['_t_end'] = vd_t['person_id'].map(t_end_map_k)
    vd_t = vd_t[
        (vd_t['date'] >= vd_t['_t0'])
        & (vd_t['date'] <= vd_t['_t_end'])
    ].drop(columns=['_t0', '_t_end'])

    # -- Filter valid_nights to [t0_k, t_end_k] ------------------------------
    vn_t = vn[vn['person_id'].isin(pid_set)].copy()
    vn_t['_t0']    = vn_t['person_id'].map(t0_map_k)
    vn_t['_t_end'] = vn_t['person_id'].map(t_end_map_k)
    vn_t = vn_t[
        (vn_t['night_date'] >= vn_t['_t0'])
        & (vn_t['night_date'] <= vn_t['_t_end'])
    ].drop(columns=['_t0', '_t_end'])
    print(f'  valid_days  in window: {len(vd_t):,} rows, {vd_t["person_id"].nunique():,} pids')
    print(f'  valid_nights in window: {len(vn_t):,} rows, {vn_t["person_id"].nunique():,} pids')

    # -- Day features ---------------------------------------------------------
    day_f = (
        vd_t.groupby('person_id', as_index=False)
        .agg(n_valid_days=('steps', 'count'), mean_steps=('steps', 'mean'))
    )

    # -- Night features (WASO, sleep duration) ---------------------------------
    ngt_f = (
        vn_t.groupby('person_id', as_index=False)
        .agg(
            n_valid_nights=('minute_asleep', 'count'),
            mean_waso=('minute_awake', 'mean'),
            mean_sleep_duration=('minute_asleep', 'mean'),
        )
    )

    # -- STV: consecutive pairs within task window ----------------------------
    vn_sorted = vn_t.sort_values(['person_id', 'night_date']).reset_index(drop=True)
    vn_sorted['night_date_prev']  = vn_sorted.groupby('person_id')['night_date'].shift(1)
    vn_sorted['onset_anch_prev']  = vn_sorted.groupby('person_id')['onset_anch'].shift(1)
    vn_sorted['offset_anch_prev'] = vn_sorted.groupby('person_id')['offset_anch'].shift(1)

    day_diff = (vn_sorted['night_date'] - vn_sorted['night_date_prev']).dt.days
    consec   = vn_sorted[day_diff == 1].copy()
    consec['d_onset']  = (consec['onset_anch']  - consec['onset_anch_prev']).abs()
    consec['d_offset'] = (consec['offset_anch'] - consec['offset_anch_prev']).abs()

    stv_f = (
        consec.groupby('person_id', as_index=False)
        .agg(
            n_consecutive_pairs=('d_onset', 'count'),
            MASD_onset=('d_onset', 'mean'),
            MASD_offset=('d_offset', 'mean'),
        )
    )
    stv_f['STV'] = np.where(
        stv_f['n_consecutive_pairs'] >= STV_PAIRS_MIN,
        (stv_f['MASD_onset'] + stv_f['MASD_offset']) / 2,
        np.nan,
    )

    # -- Assemble per-task DataFrame ------------------------------------------
    task_df = tw[['person_id', 'group', 't0', 't_end']].copy()
    task_df['task'] = task
    task_df = (
        task_df
        .merge(day_f,  on='person_id', how='left')
        .merge(ngt_f,  on='person_id', how='left')
        .merge(stv_f,  on='person_id', how='left')
    )

    for col in ['n_valid_days', 'n_valid_nights', 'n_consecutive_pairs']:
        task_df[col] = task_df[col].fillna(0).astype(int)

    task_df['wearable_eligible'] = (
        (task_df['n_valid_days']   >= N_REC_DAYS_MIN)
        & (task_df['n_valid_nights'] >= N_REC_NIGHTS_MIN)
    )

    task_df.loc[task_df['n_valid_days']   < N_REC_DAYS_MIN,   'mean_steps'] = np.nan
    task_df.loc[task_df['n_valid_nights'] < N_REC_NIGHTS_MIN, 'mean_waso']  = np.nan
    task_df.loc[task_df['n_valid_nights'] < N_REC_NIGHTS_MIN, 'mean_sleep_duration'] = np.nan

    # -- Photoperiod at the window midpoint (t0 + 45 days) --------------------
    midpoint_date = task_df['t0'] + pd.Timedelta(days=N_DAYS_WEARABLE // 2)
    jday          = midpoint_date.dt.dayofyear
    lat_vals      = task_df['person_id'].map(lat_map_k).values
    task_df['photoperiod_hours'] = daylength_vec(lat_vals, jday.values)

    results.append(task_df)

    for grp in ['MDD', 'BD', 'HC']:
        sub_g  = task_df[task_df['group'] == grp]
        if len(sub_g) == 0:
            continue
        n_elig = int(sub_g['wearable_eligible'].sum())
        n_stv  = int(sub_g['STV'].notna().sum())
        print(f'  {grp}: wearable_eligible: {n_elig:,} / {len(sub_g):,}  |  STV valid: {n_stv:,}')

    del vd_t, vn_t, day_f, ngt_f, vn_sorted, consec, stv_f, task_df
    gc.collect()

# -- Concatenate into long-format output --------------------------------------
wearable_features = pd.concat(results, ignore_index=True)
del results, valid_days, vn
gc.collect()

print(f'\nwearable_features (long format): {len(wearable_features):,} rows '
      f'({wearable_features["person_id"].nunique():,} unique participants × {len(ETM_TASKS)} tasks)')
print(f"  MDD rows: {(wearable_features['group']=='MDD').sum():,}")
print(f"  BD rows:  {(wearable_features['group']=='BD').sum():,}")
print(f"  HC rows:  {(wearable_features['group']=='HC').sum():,}")


## Task 4: Save and QC

In [ ]:
# -- Enforce column order per schema ------------------------------------------
ordered_cols = [
    'person_id', 'group', 'task', 't0', 't_end',
    'n_valid_days', 'n_valid_nights', 'wearable_eligible',
    'mean_steps', 'mean_waso', 'mean_sleep_duration',
    'MASD_onset', 'MASD_offset', 'STV', 'n_consecutive_pairs',
    'photoperiod_hours',
]
extra = [c for c in wearable_features.columns if c not in ordered_cols]
if extra:
    print(f'Unexpected extra columns (appended): {extra}')
wearable_features = wearable_features[ordered_cols + extra].reset_index(drop=True)

# -- Save ---------------------------------------------------------------------
# Always saved (not OVERWRITE-gated): the extraction above always runs.
save_parquet(wearable_features, FEATS_OUT)

print(f'\nSchema:')
print(wearable_features.dtypes.to_string())

# -- Per-task QC summary (by group) -------------------------------------------
print('\n=== Per-task feature availability ===\n')
for task in ETM_TASKS:
    sub = wearable_features[wearable_features['task'] == task]
    print(f'--- {task} ---')
    for grp in ['MDD', 'BD', 'HC']:
        sub_g = sub[sub['group'] == grp]
        n     = len(sub_g)
        if n == 0:
            continue
        n_elig  = int(sub_g['wearable_eligible'].sum())
        n_steps = int(sub_g['mean_steps'].notna().sum())
        n_waso  = int(sub_g['mean_waso'].notna().sum())
        n_sdur  = int(sub_g['mean_sleep_duration'].notna().sum())
        n_stv   = int(sub_g['STV'].notna().sum())
        n_pp    = int(sub_g['photoperiod_hours'].notna().sum())
        print(f'  {grp} (N={n:,}):')
        print(f'    wearable_eligible (n_days>=30 AND n_nights>=30) : {n_elig:,} ({100*n_elig/n:.1f}%)')
        print(f'    mean_steps        (n_valid_days >= {N_REC_DAYS_MIN})          : {n_steps:,} ({100*n_steps/n:.1f}%)')
        print(f'    mean_waso         (n_valid_nights >= {N_REC_NIGHTS_MIN})          : {n_waso:,} ({100*n_waso/n:.1f}%)')
        print(f'    mean_sleep_duration (n_valid_nights >= {N_REC_NIGHTS_MIN})        : {n_sdur:,} ({100*n_sdur/n:.1f}%)')
        print(f'    STV               (n_consec_pairs >= {STV_PAIRS_MIN})          : {n_stv:,} ({100*n_stv/n:.1f}%)')
        print(f'    photoperiod_hours (latitude non-NaN)            : {n_pp:,} ({100*n_pp/n:.1f}%)')
    print()

# -- Feature distribution summaries (all tasks pooled, non-NaN values) --------
print('=== Feature distributions (all tasks pooled, non-NaN values) ===')
for col, unit in [
    ('mean_steps',          'steps/day'),
    ('mean_waso',           'min'),
    ('mean_sleep_duration', 'min'),
    ('STV',                 'h'),
    ('MASD_onset',          'h'),
    ('MASD_offset',         'h'),
    ('photoperiod_hours',   'h'),
    ('n_consecutive_pairs', 'pairs'),
]:
    vals = wearable_features[col].dropna()
    if len(vals) == 0:
        print(f'  {col}: no non-NaN values')
        continue
    print(f'  {col} ({unit}): N={len(vals):,}  '
          f'median={vals.median():.2f}  mean={vals.mean():.2f}  sd={vals.std():.2f}  '
          f'[min={vals.min():.2f}, max={vals.max():.2f}]')

# -- Attrition log (informational; per task and group) -----------------------
for task in ETM_TASKS:
    sub = wearable_features[wearable_features['task'] == task]
    for grp in ['MDD', 'BD', 'HC']:
        sub_g    = sub[sub['group'] == grp]
        if len(sub_g) == 0:
            continue
        n_base   = len(sub_g)
        n_inelig = int((~sub_g['wearable_eligible']).sum())
        log_attrition(
            step=f'NB03b informational: wearable_eligible=False — {task} ({grp})',
            group=grp,
            n_before=n_base,
            n_after=n_base - n_inelig,
            reason=(
                f'{n_inelig} {grp} completers with < {N_REC_DAYS_MIN} valid days OR '
                f'< {N_REC_NIGHTS_MIN} valid nights within '
                f'[etm_date_{task} − {N_DAYS_WEARABLE}d, etm_date_{task} − 1d]. '
                'Retained in wearable_features; wearable_eligible=False excludes them '
                'from the wearable analyses.'
            ),
            task=task,
        )

atr_df = pd.DataFrame(attrition_rows)
save_parquet(atr_df, 'cohort/attrition.parquet')
print(f'\nAttrition log updated ({len(atr_df)} total entries) → cohort/attrition.parquet')